<a href="https://colab.research.google.com/github/slavyolov/Algorithms/blob/main/Walk_Forward_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.3 MB/s eta 0:00:00


### Create the Data

In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from loguru import logger

import os
import pandas as pd
import numpy as np

In [2]:

# Set a random seed for reproducibility
np.random.seed(42)

# Create a dummy weather.csv file for demonstration
dummy_data = {
    'date': pd.to_datetime(pd.date_range(start='2022-01-01', periods=400, freq='D')),
    'temp': np.random.randint(260, 310, size=400) / 10 # Temperatures in Kelvin (approx 26-31 Celsius)
}
df = pd.DataFrame(dummy_data)

print("Dummy weather data created.")

Dummy weather data created.


### Model fitting with Expanding-Window Walkforward Validation

In [6]:
# Create simple lag features
n_lags = 7
for lag in range(1, n_lags + 1):
    df[f'lag_{lag}'] = df['temp'].shift(lag)
df_currated = df.dropna().reset_index(drop=True).copy(deep=True)

# Parameters:
initial_train_days = 365   # start with one year
test_block = 7             # evaluate weekly blocks
X_cols = [c for c in df.columns if c.startswith('lag_')]
y_col = 'temp'

logger.info(f"Features : {X_cols}")

# Data Containers
preds_all, trues_all = [], []
window_rmse = []

# Random Forest Regressor trained using Expanding Walk-Forward Validation
train_end = initial_train_days
while train_end + test_block <= len(df_currated):
    train = df_currated.iloc[:train_end]
    test = df_currated.iloc[train_end: train_end + test_block]

    logger.info(f"Processing window: train_end={train_end}, test_block={test_block}")

    # Fit scaler and model on train only
    logger.info(f"Applying StandardScaler to training data (rows 0 to {train_end-1})")
    scaler = StandardScaler().fit(train[X_cols])
    X_train = scaler.transform(train[X_cols])
    X_test = scaler.transform(test[X_cols])

    model = RandomForestRegressor(n_estimators=100, random_state=0)
    logger.info(f"Training RandomForestRegressor on data from row 0 to {train_end-1}")
    model.fit(X_train, train[y_col].values)

    y_pred = model.predict(X_test)
    y_true = test[y_col].values

    preds_all.extend(y_pred.tolist())
    trues_all.extend(y_true.tolist())
    window_rmse.append(np.sqrt(mean_squared_error(y_true, y_pred)))

    # expanding window: add the test block to training for next iteration
    train_end += test_block

# Summary metrics
overall_rmse = np.sqrt(mean_squared_error(trues_all, preds_all))
logger.info("\n" + "="*100)
logger.info(f"Expanding-window WFV overall RMSE: {overall_rmse:.3f}")
logger.info(
    "Per-window RMSEs:\n{}",
    pd.DataFrame(window_rmse).to_string(index=False)
)

2026-05-25 16:10:23.048 | INFO     | __main__:<cell line: 0>:22 - Features : ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7']
2026-05-25 16:10:23.050 | INFO     | __main__:<cell line: 0>:34 - Processing window: train_end=365, test_block=7
2026-05-25 16:10:23.052 | INFO     | __main__:<cell line: 0>:37 - Applying StandardScaler to training data (rows 0 to 364)
2026-05-25 16:10:23.065 | INFO     | __main__:<cell line: 0>:43 - Training RandomForestRegressor on data from row 0 to 364
2026-05-25 16:10:23.551 | INFO     | __main__:<cell line: 0>:34 - Processing window: train_end=372, test_block=7
2026-05-25 16:10:23.552 | INFO     | __main__:<cell line: 0>:37 - Applying StandardScaler to training data (rows 0 to 371)
2026-05-25 16:10:23.561 | INFO     | __main__:<cell line: 0>:43 - Training RandomForestRegressor on data from row 0 to 371
2026-05-25 16:10:23.877 | INFO     | __main__:<cell line: 0>:34 - Processing window: train_end=379, test_block=7
2026-05-25 16:10:23.878 | IN

In [4]:
window_rmse

[np.float64(1.2998497715395585),
 np.float64(1.754442646540496),
 np.float64(1.3385540172674597),
 np.float64(1.3377890074938636)]

## Bonus part, just mentioned in the article

In [ ]:
# Walk-forward (expanding) example — one-step daily temperature forecast with inner GridSearchCV
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV # Import GridSearchCV
from loguru import logger

# Replace with your CSV that has 'date' and 'temp' columns
df = pd.read_csv('weather.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)

# Create simple lag features
n_lags = 7
for lag in range(1, n_lags + 1):
    df[f'lag_{lag}'] = df['temp'].shift(lag)
df = df.dropna().reset_index(drop=True)

# Parameters: tune these to match your use case
initial_train_days = 365   # start with one year
test_block = 7             # evaluate weekly blocks
X_cols = [c for c in df.columns if c.startswith('lag_')]
y_col = 'temp'

preds_all, trues_all = [], []
window_rmse = []

# Define the parameter grid for RandomForestRegressor (can be adjusted)
# This is defined once and used for GridSearchCV in each window
param_grid = {
    'n_estimators': [50, 100], # Reduced for faster execution in a loop
    'max_depth': [None, 10],   # Reduced for faster execution
    'min_samples_split': [2, 5]
}

train_end = initial_train_days
while train_end + test_block <= len(df):
    train = df.iloc[:train_end]
    test = df.iloc[train_end: train_end + test_block]

    logger.info(f"\nProcessing window: train_end={train_end}, test_block={test_block}")

    # Fit scaler on the current training data
    logger.info(f"Applying StandardScaler to training data (rows 0 to {train_end-1})")
    scaler = StandardScaler().fit(train[X_cols])
    X_train = scaler.transform(train[X_cols])
    X_test = scaler.transform(test[X_cols])

    # --- Hyperparameter tuning within the current window ---
    logger.info("Starting GridSearchCV for hyperparameter tuning on current training window...")
    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=0),
        param_grid=param_grid,
        cv=2, # Reduced cv for faster execution in a loop
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=0 # Suppress verbose output from GridSearchCV itself
    )

    grid_search.fit(X_train, train[y_col].values)
    best_model = grid_search.best_estimator_
    logger.info(f"GridSearchCV completed. Best parameters: {grid_search.best_params_}")

    # --- Prediction using the best model from tuning ---
    logger.info(f"Predicting with best RandomForestRegressor model on test data (rows {train_end} to {train_end + test_block - 1})")
    y_pred = best_model.predict(X_test)
    y_true = test[y_col].values

    preds_all.extend(y_pred.tolist())
    trues_all.extend(y_true.tolist())
    window_rmse.append(np.sqrt(mean_squared_error(y_true, y_pred)))

    # expanding window: add the test block to training for next iteration
    train_end += test_block

# Summary metrics
overall_rmse = np.sqrt(mean_squared_error(trues_all, preds_all))
logger.info(f'\nExpanding-window WFV (with inner GridSearchCV) overall RMSE: {overall_rmse:.3f}')
logger.info('Per-window RMSEs:\n', pd.DataFrame(window_rmse))

2026-05-14 22:30:32.361 | INFO     | __main__:<cell line: 0>:41 - 
Processing window: train_end=365, test_block=7
2026-05-14 22:30:32.362 | INFO     | __main__:<cell line: 0>:44 - Applying StandardScaler to training data (rows 0 to 364)
2026-05-14 22:30:32.371 | INFO     | __main__:<cell line: 0>:50 - Starting GridSearchCV for hyperparameter tuning on current training window...
2026-05-14 22:30:38.368 | INFO     | __main__:<cell line: 0>:62 - GridSearchCV completed. Best parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 50}
2026-05-14 22:30:38.369 | INFO     | __main__:<cell line: 0>:65 - Predicting with best RandomForestRegressor model on test data (rows 365 to 371)
2026-05-14 22:30:38.378 | INFO     | __main__:<cell line: 0>:41 - 
Processing window: train_end=372, test_block=7
2026-05-14 22:30:38.379 | INFO     | __main__:<cell line: 0>:44 - Applying StandardScaler to training data (rows 0 to 371)
2026-05-14 22:30:38.387 | INFO     | __main__:<cell line: 0>:50 - S